# Phase 2 — Section 1 Report
## Database Implementation and Data Querying

**Dataset:** Crawled Stack Overflow C++ Questions Dataset  
**Selected final task:** Semantic similar-question recommendation system  
**Database:** SQLite

This is the complete, self-contained report and implementation for Section 1. It documents the database decision, creates the relational schema, imports the CSV, verifies data integrity, and runs SQL queries.

## 1. Project decision

We selected a **semantic similar-question recommendation system**. Each question has a rich title and body, which can later be converted to TF-IDF vectors or embeddings and used to retrieve related historical questions.

This is more appropriate than the other proposed options:

- **Difficulty prediction:** the dataset has no real difficulty label. Creating one from votes or views would introduce an artificial and potentially biased target.
- **Auto-tagging:** the data contains 1,691 distinct tags across only 2,500 questions, making this a sparse, imbalanced multi-label classification problem.
- **Similar-question recommendation:** directly uses the available textual fields and requires no invented target label.

SQLite was chosen because the dataset is local and modest in size, the project needs a reproducible relational database, and SQLite requires no separate server while supporting keys, indexes, joins, and standard SQL queries.

## 2. Running the database implementation

Run the notebook cells from top to bottom (or select **Run All**). The import cell reads the original CSV, creates the SQLite database, and imports the normalized tables.

If needed, install the two dependencies once from the `section1_database` folder:

```powershell
python -m pip install -r requirements.txt
```

The import cell validates the source CSV, converts Stack Exchange Unix timestamps to ISO-8601 UTC, normalizes users and tags, and creates `data/stackoverflow_questions.db`. Re-running the notebook recreates the database deterministically.

In [1]:
from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the project root (works whether the notebook is started from notebooks/,
# section1_database/, or the workspace root) and import the shared pipeline code so
# this report stays in sync with scripts/import_to_db.py (single source of truth).
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents, cwd / 'Phase 2' / 'section1_database']
PROJECT_ROOT = next(
    (candidate for candidate in candidate_roots
     if (candidate / 'requirements.txt').exists() and (candidate / 'pipeline.py').exists()),
    cwd,
)
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.database_connection import SOURCE_CSV_PATH, get_database_path
from scripts.import_to_db import SCHEMA_SQL, build_database

CSV_PATH = SOURCE_CSV_PATH

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Source CSV not found at: {CSV_PATH}')

print(f'Source CSV: {CSV_PATH}')
print(f'Project root: {PROJECT_ROOT}')

Source CSV: /home/user/Data-Science/Phase 1/stackoverflow_questions.csv
Project root: /home/user/Data-Science/Phase 2/Project_P2


## 3. Relational schema

```text
users (1) ──< questions (1) ──< question_tags >── (1) tags
```

| Table | Primary key | Purpose |
| --- | --- | --- |
| `users` | `user_id` | Stores one record for every known question owner. Some questions have no owner because their owner is deleted/unregistered. |
| `questions` | `question_id` | Stores the question text, URL, engagement metrics, answer/closure state, and timestamps. `owner_user_id` is a nullable foreign key to `users`. |
| `tags` | `tag_id` | Stores every distinct tag once. `name` is unique, case-insensitively. |
| `question_tags` | (`question_id`, `tag_id`) | Junction table for the many-to-many relationship between questions and tags. |

The schema is normalized: owner details are not duplicated across questions, and a list of tags is not stored as one text field. The next cell contains the complete DDL, integrity constraints, and indexes.

In [2]:
# The full DDL lives in scripts/import_to_db.py and is reused here to avoid drift.
print(SCHEMA_SQL)


PRAGMA foreign_keys = ON;

CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    account_id INTEGER UNIQUE,
    reputation INTEGER,
    user_type TEXT NOT NULL,
    display_name TEXT NOT NULL,
    accept_rate REAL CHECK (accept_rate BETWEEN 0 AND 100),
    profile_image_url TEXT,
    profile_url TEXT
);

CREATE TABLE questions (
    question_id INTEGER PRIMARY KEY,
    owner_user_id INTEGER,
    title TEXT NOT NULL,
    body_html TEXT NOT NULL,
    question_url TEXT NOT NULL UNIQUE,
    content_license TEXT,
    is_answered INTEGER NOT NULL CHECK (is_answered IN (0, 1)),
    view_count INTEGER NOT NULL CHECK (view_count >= 0),
    answer_count INTEGER NOT NULL CHECK (answer_count >= 0),
    score INTEGER NOT NULL,
    accepted_answer_id INTEGER,
    creation_at TEXT NOT NULL,
    last_activity_at TEXT NOT NULL,
    last_edit_at TEXT,
    closed_at TEXT,
    closed_reason TEXT,
    bounty_amount INTEGER CHECK (bounty_amount >= 0),
    bounty_closes_at TEXT,
    protected_at TEXT,
 

In [3]:
# Rebuild the normalized database from the CSV using the shared pipeline function.
DB_PATH, import_counts = build_database()
print(f'Database created at: {DB_PATH}')
pd.DataFrame(import_counts.items(), columns=['imported_table', 'row_count'])

Database created at: /home/user/Data-Science/Phase 2/Project_P2/data/stackoverflow_questions.db


,imported_table,row_count
0,users,2056
1,questions,2500
2,tags,1691
3,question_tags,8598


### Source-to-schema mapping

| Source CSV fields | Database destination |
| --- | --- |
| `question_id`, `title`, `body`, `link`, status/engagement fields | `questions` |
| `owner.*` fields | `users`; `owner.user_id` also becomes `questions.owner_user_id` |
| `tags` (a list in the CSV) | Parsed into `tags` and `question_tags` |
| Unix epoch date columns | ISO-8601 UTC timestamp columns in `questions` |

The `migrated_from.other_site.*` fields are present for only four records and are not useful for the selected NLP task, so they are intentionally not included in the database. The original CSV remains unchanged as the source of record.

## 4. Import and integrity verification

The last successful import produced: **2,500 questions**, **2,056 users**, **1,691 tags**, and **8,598 question–tag relationships**. Every question has at least one tag, and the foreign-key check returned zero violations.

In [4]:
with sqlite3.connect(DB_PATH) as connection:
    integrity_results = {
        'questions': connection.execute('SELECT COUNT(*) FROM questions').fetchone()[0],
        'users': connection.execute('SELECT COUNT(*) FROM users').fetchone()[0],
        'tags': connection.execute('SELECT COUNT(*) FROM tags').fetchone()[0],
        'question_tag_links': connection.execute('SELECT COUNT(*) FROM question_tags').fetchone()[0],
        'questions_without_tags': connection.execute(
            """
            SELECT COUNT(*)
            FROM questions AS q
            LEFT JOIN question_tags AS qt ON qt.question_id = q.question_id
            WHERE qt.question_id IS NULL
            """
        ).fetchone()[0],
        'foreign_key_violations': len(connection.execute('PRAGMA foreign_key_check').fetchall()),
    }

pd.DataFrame(integrity_results.items(), columns=['check', 'value'])

,check,value
0,questions,2500
1,users,2056
2,tags,1691
3,question_tag_links,8598
4,questions_without_tags,0
5,foreign_key_violations,0


## 5. SQL query results

The following queries demonstrate that the data can be queried through the normalized schema. They are included directly in the executable cells below.

In [5]:
overview_sql = """
SELECT
    (SELECT COUNT(*) FROM questions) AS questions,
    (SELECT COUNT(*) FROM users) AS users,
    (SELECT COUNT(*) FROM tags) AS tags,
    (SELECT COUNT(*) FROM question_tags) AS question_tag_links,
    ROUND(1.0 * (SELECT COUNT(*) FROM question_tags) /
          (SELECT COUNT(*) FROM questions), 2) AS avg_tags_per_question,
    MIN(creation_at) AS earliest_question,
    MAX(creation_at) AS latest_question
FROM questions;
"""

with sqlite3.connect(DB_PATH) as connection:
    display(pd.read_sql_query(overview_sql, connection))

,questions,users,tags,question_tag_links,avg_tags_per_question,earliest_question,latest_question
0,2500,2056,1691,8598,3.44,2008-08-01T12:13:50Z,2026-05-28T16:11:59Z


**Interpretation.** The database contains 2,500 questions, 2,056 known owners, 1,691 distinct tags, and 8,598 question–tag links (3.44 tags per question on average). The questions span from 2008 to 2026, providing a varied historical corpus for retrieval. The relatively high number of distinct tags versus questions also confirms that auto-tagging would be a sparse multi-label task, while text-based recommendation is well supported.

In [6]:
top_tags_sql = """
SELECT t.name AS tag, COUNT(*) AS question_count
FROM tags AS t
JOIN question_tags AS qt ON qt.tag_id = t.tag_id
GROUP BY t.tag_id, t.name
ORDER BY question_count DESC, t.name ASC
LIMIT 10;
"""

with sqlite3.connect(DB_PATH) as connection:
    display(pd.read_sql_query(top_tags_sql, connection))

,tag,question_count
0,c++,2500
1,c,128
2,c++11,117
3,qt,105
4,windows,104
5,templates,102
6,cmake,101
7,language-lawyer,98
8,c++20,89
9,winapi,89


**Interpretation.** Every collected record has the `c++` tag, which is expected because the crawl targeted C++ questions. The next most common tags (`c`, `c++11`, `qt`, `windows`, `templates`, and `cmake`) reveal useful technical subtopics. These tags can be used as optional metadata filters for recommendations, but the long tail of rare tags should not be treated as a reliable supervised label without much more data.

In [7]:
top_questions_sql = """
SELECT question_id, title, score, view_count, answer_count, creation_at
FROM questions
ORDER BY score DESC, question_id ASC
LIMIT 10;
"""

with sqlite3.connect(DB_PATH) as connection:
    display(pd.read_sql_query(top_questions_sql, connection))

,question_id,title,score,view_count,answer_count,creation_at
0,11227809,Why is conditional processing of a sorted arra...,27530,1985154,26,2012-06-27T13:51:36Z
1,4421706,What are the basic rules and idioms for operat...,2494,1065181,10,2010-12-12T12:44:56Z
2,106508,What is a smart pointer and when should I use ...,2196,776606,14,2008-09-20T00:09:24Z
3,5590381,How can I convert int to string in C++?,2194,5131807,23,2011-04-08T04:19:41Z
4,12135518,Is &lt; faster than &lt;=?,1786,160601,15,2012-08-27T02:10:12Z
5,356950,What are C++ functors and their uses?,1104,633704,14,2008-12-10T17:47:21Z
6,98650,What is the strict aliasing rule?,1038,335325,11,2008-09-19T01:30:27Z
7,216823,How can I trim a std::string?,1035,1112484,52,2008-10-19T19:23:07Z
8,612328,Difference between &#39;struct&#39; and &#39;t...,983,687631,7,2009-03-04T20:41:12Z
9,154136,Why use apparently meaningless do-while and if...,969,125519,9,2008-09-30T17:36:24Z


**Interpretation.** Scores and views are strongly right-skewed: a small number of classic questions have exceptionally high engagement. Therefore, if engagement is later used for tie-breaking or ranking, it should be transformed with `log1p` and scaled rather than used in raw form. Engagement is not used as the semantic-similarity target, because popularity does not necessarily mean that two questions are technically similar.

In [8]:
# This tag-based query is a simple metadata pre-filter for the future recommender.
candidate_sql = """
SELECT q.question_id, q.title, q.score, q.creation_at
FROM questions AS q
JOIN question_tags AS qt ON qt.question_id = q.question_id
JOIN tags AS t ON t.tag_id = qt.tag_id
WHERE t.name = 'c++' AND q.is_answered = 1
ORDER BY q.score DESC, q.creation_at DESC
LIMIT 10;
"""

with sqlite3.connect(DB_PATH) as connection:
    display(pd.read_sql_query(candidate_sql, connection))

,question_id,title,score,creation_at
0,11227809,Why is conditional processing of a sorted arra...,27530,2012-06-27T13:51:36Z
1,4421706,What are the basic rules and idioms for operat...,2494,2010-12-12T12:44:56Z
2,106508,What is a smart pointer and when should I use ...,2196,2008-09-20T00:09:24Z
3,5590381,How can I convert int to string in C++?,2194,2011-04-08T04:19:41Z
4,12135518,Is &lt; faster than &lt;=?,1786,2012-08-27T02:10:12Z
5,356950,What are C++ functors and their uses?,1104,2008-12-10T17:47:21Z
6,98650,What is the strict aliasing rule?,1038,2008-09-19T01:30:27Z
7,216823,How can I trim a std::string?,1035,2008-10-19T19:23:07Z
8,612328,Difference between &#39;struct&#39; and &#39;t...,983,2009-03-04T20:41:12Z
9,154136,Why use apparently meaningless do-while and if...,969,2008-09-30T17:36:24Z


**Interpretation.** This query demonstrates a valid relational join and a simple candidate pre-filter: it retrieves answered C++ questions, ranked by community score. It is not the final recommender by itself; the pipeline in Section 3 will create TF-IDF features from the title and body and use cosine similarity to rank candidates by meaning. Tags and score remain useful optional filters/tie-breakers.

## 6. Conclusion

Section 1 is complete. The original flat CSV has been imported into a reproducible, normalized SQLite database with data-integrity constraints and useful indexes. The `questions` table preserves the text needed for semantic recommendation, while the normalized tag structure supports SQL filtering and later evaluation.

The next project phase can load the question text directly from this database for preprocessing and feature engineering.